In [1]:
import numpy as np

In [ ]:
def initialize_parameters(layer_dims, seed = 42):
    np.random.seed(seed)
    parameters = {}
    L = len(layer_dims) - 1
    for l in range(1, L+1):
        parameters[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2. / layer_dims[l-1])
        parameters[f'b{l}'] = np.zeros((layer_dims[l], 1))
    return parameters

In [3]:
def linear_forward(A_prev, W, b):
    Z = W.dot(A_prev) + b
    cache = (A_prev, W, b)
    return Z, cache

In [4]:
def relu(Z):
    A = np.maximum(0, Z)
    cache = Z
    return A, cache

In [5]:
def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    A = expZ / np.sum(expZ, axis=0, keepdims=True)
    cache = Z
    return A, cache

In [14]:
def forward_propagation(X, parameters):
    caches = []
    A = X
    L = len(parameters) // 2  # number of layers

    # hidden layers
    for l in range(1, L):
        A_prev = A
        Z, linear_cache = linear_forward(A_prev, parameters[f'W{l}'], parameters[f'b{l}'])
        A, activation_cache = relu(Z)
        caches.append((linear_cache, activation_cache))

    # output layer
    ZL, linear_cache = linear_forward(A, parameters[f'W{L}'], parameters[f'b{L}'])
    AL, activation_cache = softmax(ZL)
    caches.append((linear_cache, activation_cache))

    return AL, caches

In [9]:
def compute_cost(AL, Y):
    m = Y.shape[1]
    cost = -np.sum(Y * np.log(AL + 1e-8)) / m
    return np.squeeze(cost)

In [10]:
def linear_backward(dZ, cache):
    A_prev, W, b = cache
    m = A_prev.shape[1]
    dW = (1. / m) * dZ.dot(A_prev.T)
    db = (1. / m) * np.sum(dZ, axis=1, keepdims=True)
    dA_prev = W.T.dot(dZ)
    return dA_prev, dW, db

In [11]:
def relu_backward(dA, cache):
    Z = cache
    dZ = np.array(dA, copy=True)
    dZ[Z <= 0] = 0
    return dZ

In [12]:
def backward_propagation(AL, Y, caches):
    grads = {}
    L = len(caches)  # total layers
    m = AL.shape[1]
    Y = Y.reshape(AL.shape)

    # output layer gradient
    linear_cache, activation_cache = caches[L-1]
    dZL = AL - Y
    dA_prev, dW, db = linear_backward(dZL, linear_cache)
    grads[f'dW{L}'] = dW
    grads[f'db{L}'] = db
    grads[f'dA{L-1}'] = dA_prev

    # hidden layers gradients
    for l in reversed(range(L-1)):
        linear_cache, activation_cache = caches[l]
        dA_curr = grads[f'dA{l+1}']
        dZ = relu_backward(dA_curr, activation_cache)
        dA_prev, dW, db = linear_backward(dZ, linear_cache)
        grads[f'dW{l+1}'] = dW
        grads[f'db{l+1}'] = db
        grads[f'dA{l}'] = dA_prev

    return grads

In [15]:
# Example usage:
if __name__ == "__main__":
    # Define network structure
    layer_dims = [5, 4, 3]  # 5-input, one hidden layer of size 4, 3-output
    X = np.random.randn(5, 10)
    Y = np.eye(3)[np.random.choice(3, 10)].T

    params = initialize_parameters(layer_dims)
    AL, caches = forward_propagation(X, params)
    cost = compute_cost(AL, Y)
    grads = backward_propagation(AL, Y, caches)
    print("Cost:", cost)
    for key, val in grads.items():
        print(f"{key}.shape = {val.shape}")

Cost: 1.0136777157511605
dW2.shape = (3, 4)
db2.shape = (3, 1)
dA1.shape = (4, 10)
dW1.shape = (4, 5)
db1.shape = (4, 1)
dA0.shape = (5, 10)
